# CDLE: Diagnóstico e Profiling de Performance em Big Data

Este notebook é dedicado exclusivamente ao **profiling de CPU** das operações mais exigentes do pipeline (**Value Counts** e **GroupBy**) nas 5 frameworks: **Pandas, Dask, PySpark, Modin e Joblib**.

Utiliza-se a magia do Jupyter `%%prun -s tottime -l 15` no topo de cada célula de computação para listar e ordenar as 15 funções internas que mais consomem tempo de CPU, permitindo analisar bottlenecks de forma isolada e económica no Dataproc.

In [ ]:
import sys
!{sys.executable} -m pip install -q "modin[ray]" matplotlib

In [ ]:
# --- PATCH GLOBAL DE VISUALIZAÇÃO DE TABELAS ---
import pandas as pd
if not hasattr(pd.Index, '_format_flat'):
    pd.Index._format_flat = lambda self, *args, **kwargs: [str(x) for x in self]

try:
    import pyspark.pandas as ps
    if not hasattr(ps.Index, '_format_flat'):
        ps.Index._format_flat = lambda self, *args, **kwargs: [str(x) for x in self]
except Exception:
    pass
# -----------------------------------------------

import time
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
print(" 1 - Escala Pequena (~123k linhas)")
print(" 2 - Escala Média (~2.46M linhas)")
print(" 3 - Escala Grande (~9M linhas)")

while True:
    try:
        op = input("Introduza a opcao desejada (1, 2 ou 3): ").strip()
        ESCALA_DATASET = int(op)
        if ESCALA_DATASET in [1, 2, 3]:
            break
        else:
            print("Opcao invalida. Por favor, introduza 1, 2 ou 3.")
    except ValueError:
        print("Entrada invalida. Por favor, introduza um numero.")

base_dir = "gs://dataproc-staging-europe-southwest1-348488791616-f80l4tzf/notebooks/jupyter/"
caminho = {
    1: base_dir + "yellow_tripdata_2022_small.parquet",
    2: base_dir + "yellow_tripdata_2022-01.parquet",
    3: base_dir + "yellow_tripdata_2022_large.parquet"
}

file_path = caminho[ESCALA_DATASET]

pandas_results = {}
dask_results = {}
pyspark_results = {}
modin_results = {}
joblib_results = {}

---
## Secção 1: Profiling com Pandas

O Pandas executa em modo **eager** e single-thread. Espera-se que a maior parte do tempo de CPU seja despendida diretamente em funções internas compiladas do NumPy ou do interpretador C.

In [ ]:
df_pd = pd.read_parquet(file_path)

## 1.1 Value Counts

In [ ]:
%%prun -s tottime -l 15

import time
start = time.time()
vc_pd = df_pd['VendorID'].value_counts()
pandas_results['Value Counts'] = time.time() - start


## 1.2 GroupBy

In [ ]:
%%prun -s tottime -l 15

import time
start = time.time()
gb_pd = df_pd.groupby('payment_type')['fare_amount'].mean()
pandas_results['GroupBy'] = time.time() - start

Para limpar a memória, para evitar qualquer tipo de crash dos clusters.

In [ ]:
import gc
del df_pd, vc_pd, gb_pd
gc.collect()

---
## Secção 2: Profiling com Dask

O Dask constrói um grafo de tarefas e executa-as em chunks. O profiling revelará o tempo de coordenação do agendador síncrono e a concatenação dos pedaços intermédios.

In [ ]:
# Patch de compatibilidade Dask e Python 3.11
try:
    import dask.utils
    original_derived_from = dask.utils.derived_from
    def safe_derived_from(*args, **kwargs):
        decorator = original_derived_from(*args, **kwargs)
        def safe_decorator(func):
            try:
                return decorator(func)
            except Exception:
                return func
        return safe_decorator
    dask.utils.derived_from = safe_derived_from
    import dask
    dask.config.set(scheduler='synchronous')
except Exception as e:
    print(f"Aviso Dask patch: {e}")

import dask.dataframe as dd

df_dd = dd.read_parquet(file_path)

## 2.1 Value Count

In [ ]:
%%prun -s tottime -l 15

import time
start = time.time()
vc_dd = df_dd['VendorID'].value_counts().compute()
dask_results['Value Counts'] = time.time() - start

## 2.2 GroupBy

In [ ]:
%%prun -s tottime -l 15

import time
start = time.time()
gb_dd = df_dd.groupby('payment_type')['fare_amount'].mean().compute()
dask_results['GroupBy'] = time.time() - start

Para limpar a memória, para evitar qualquer tipo de crash dos clusters

In [ ]:
import gc
del df_dd, vc_dd, gb_dd
gc.collect()

---
## Secção 3: Profiling com PySpark (Koalas)

No PySpark, a computação real ocorre na JVM (Java Virtual Machine). O profiling do Python mostrará predominantemente chamadas de sockets, comunicação via rede, e serialização de dados através do protocolo Py4J.

In [ ]:
import os
os.environ['PYARROW_IGNORE_TIMEZONE'] = '1'
from pyspark.sql import SparkSession
import pyspark.pandas as ps

spark = SparkSession.builder \
    .appName('CDLE-Profiling') \
    .config('spark.sql.ansi.enabled', 'false') \
    .getOrCreate()

sdf = spark.read.parquet(file_path)
for col_name, col_type in sdf.dtypes:
    if 'timestamp' in col_type or 'ntz' in col_type.lower():
        sdf = sdf.withColumn(col_name, sdf[col_name].cast('timestamp'))

df_ps = sdf.to_pandas_on_spark()

## 3.1 Value Counts

In [ ]:
%%prun -s tottime -l 15

import time
start = time.time()
vc_ps = df_ps['VendorID'].value_counts()
pyspark_results['Value Counts'] = time.time() - start


## 3.2 GroupBy

In [ ]:
%%prun -s tottime -l 15

import time
start = time.time()
gb_ps = df_ps.groupby('payment_type')['fare_amount'].mean()
pyspark_results['GroupBy'] = time.time() - start


In [ ]:
import gc
spark.stop()
del df_ps, vc_ps, gb_ps
gc.collect()

---
## Secção 4: Profiling com Modin

O Modin abstrai a distribuição do Pandas. Ao correr com o engine em modo python/single-process para evitar picos de memória, o profiling revelará os wrappers de metadados e distribuição criados pelo Modin.

In [ ]:
import os
os.environ["MODIN_ENGINE"] = "python"

# Patch de compatibilidade com Pandas 2.1.4
import sys
import pandas as pd
try:
    import pandas.core.arrays.arrow
except ImportError:
    import types
    sys.modules['pandas.core.arrays.arrow'] = types.ModuleType('pandas.core.arrays.arrow')
    import pandas.core.arrays.arrow

if not hasattr(pandas.core.arrays.arrow, 'ListAccessor'):
    class DummyListAccessor: pass
    pandas.core.arrays.arrow.ListAccessor = DummyListAccessor

if not hasattr(pandas.core.arrays.arrow, 'StructAccessor'):
    class DummyStructAccessor: pass
    pandas.core.arrays.arrow.StructAccessor = DummyStructAccessor

import modin.pandas as mpd

df_mod = mpd.read_parquet(file_path)

## 4.1 Value Counts

In [ ]:
%%prun -s tottime -l 15

import time
start = time.time()
val_mod = df_mod['VendorID'].value_counts()
modin_results['Value Counts'] = time.time() - start

## 4.2 GroupBy

In [ ]:
%%prun -s tottime -l 15

import time
start = time.time()
gb_mod = df_mod.groupby('payment_type')['fare_amount'].mean()
modin_results['GroupBy'] = time.time() - start


In [ ]:
import gc
del df_mod, val_mod, gb_mod
gc.collect()

---
## Secção 5: Profiling com Joblib

O Joblib fatia o dataframe e distribui a carga por subprocessos. O profiling do processo pai mostrará o overhead de spawn/fork dos processos, serialização de dados (dumping/pickling) e coordenação da pool de processos.

In [ ]:
import pandas as pd
import numpy as np
from joblib import Parallel, delayed

df_pd_job = pd.read_parquet(file_path)

## 5.1 Value Counts

In [ ]:
%%prun -s tottime -l 15

import time
start = time.time()

def val_chunk(chunk):
    return chunk['VendorID'].value_counts()

chunks = np.array_split(df_pd_job, 4)
results = Parallel(n_jobs=4)(delayed(val_chunk)(chunk) for chunk in chunks)
vc_job = pd.concat(results).groupby(level=0).sum()
joblib_results['Value Counts'] = time.time() - start

## 5.2 GroupBy

In [ ]:
%%prun -s tottime -l 15

import time
start = time.time()

def gb_chunk(chunk):
    return chunk.groupby('payment_type')['fare_amount'].agg(['sum', 'count'])
    
chunks = np.array_split(df_pd_job, 4)
results = Parallel(n_jobs=4)(delayed(gb_chunk)(chunk) for chunk in chunks)
combined = pd.concat(results).groupby('payment_type').sum()
gb_job = combined['sum'] / combined['count']
joblib_results['GroupBy'] = time.time() - start


In [ ]:
import gc
del df_pd_job, vc_job, gb_job
gc.collect()

---
## Secção 6: Análise Comparativa das Operações sob Instrumentação

O bloco abaixo reúne os tempos de execução recolhidos *dentro* do ambiente de profiling. Note que estes tempos incluem o overhead introduzido pelo `cProfile`, servindo para analisar a relação de performance das operações complexas entre si.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

if 'ESCALA_DATASET' not in locals():
    ESCALA_DATASET = 1

escala_nome = {1: 'Pequena', 2: 'Média', 3: 'Grande'}.get(ESCALA_DATASET, 'Pequena')

active_profiling = {}
try:
    if 'pandas_results' in locals() and pandas_results:
        active_profiling['Pandas (s)'] = pandas_results
except NameError: pass

try:
    if 'dask_results' in locals() and dask_results:
        active_profiling['Dask (s)'] = dask_results
except NameError: pass

try:
    if 'pyspark_results' in locals() and pyspark_results:
        active_profiling['PySpark (s)'] = pyspark_results
except NameError: pass

try:
    if 'modin_results' in locals() and modin_results:
        active_profiling['Modin (s)'] = modin_results
except NameError: pass

try:
    if 'joblib_results' in locals() and joblib_results:
        active_profiling['Joblib (s)'] = joblib_results
except NameError: pass

if active_profiling:
    prof_df = pd.DataFrame(active_profiling)
    
    print('='*60)
    print(f'  TEMPOS DE EXECUÇÃO SOB PROFILING - ESCALA {escala_nome.upper()}  ')
    print('='*60)
    display(prof_df.round(3))
    
    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
    fig, ax = plt.subplots(figsize=(12, 6))
    
    import numpy as np
    x = np.arange(len(prof_df))
    cols = prof_df.columns
    width = 0.7 / max(1, len(cols))
    cmap = plt.colormaps.get_cmap('viridis')
    
    for i, col in enumerate(cols):
        color_val = cmap(i / max(1, len(cols) - 1)) if len(cols) > 1 else cmap(0.3)
        offset = (i - (len(cols) - 1) / 2.0) * width
        ax.bar(x + offset, prof_df[col], width, label=col, color=color_val, edgecolor='black', alpha=0.9)
        
    ax.set_xticks(x)
    ax.set_xticklabels(prof_df.index)
    
    plt.title(f'Performance de Operações sob Profiling - Escala {escala_nome} (Com Overhead do cProfile)', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Operações Analisadas', fontsize=11, fontweight='bold', labelpad=10)
    plt.ylabel('Tempo de Execução (segundos)', fontsize=11, fontweight='bold', labelpad=10)
    plt.xticks(rotation=0, fontsize=10)
    plt.yticks(fontsize=10)
    ax.yaxis.grid(True, linestyle='--', alpha=0.6)
    
    plt.legend(title='Frameworks Monitorizados', frameon=True, shadow=True, facecolor='white')
    plt.tight_layout()
    
    plt.show()

# Relatório de Análise e Diagnóstico Comparativo de Desempenho (CPU)

Este capítulo apresenta um estudo analítico e comparativo entre cinco das principais tecnologias de manipulação de dados em Python (Pandas, Dask, PySpark, Modin e Joblib).

* **Fase de Diagnóstico (Com Profiling)**: Focada na interceção detalhada das chamadas de funções a nível do interpretador (através do utilitário `cProfile`) para mapear e isolar os gargalos físicos e lógicos de cada arquitetura.

---

## 1. Fase de Diagnóstico (Com Profiling)

Através do utilitário `cProfile`, analisou-se o comportamento das bibliotecas a nível de funções internas do sistema durante as operações de agregação (`GroupBy`) e frequências (`Value Counts`) sob a Escala Média (~2.46M linhas).

### 2.1. Isolamento de Gargalos por Tecnologia
* **PySpark (Pandas-on-Spark)**: O maior tempo de CPU em Python foi consumido no método `{method 'recv_into' of '_socket.socket' objects}`. Isto ocorre porque a API em Python do Spark funciona como um cliente leve que comunica com a JVM via sockets TCP locais (`Py4J`). O interpretador de Python não realiza processamento físico; ele fica bloqueado a aguardar pela resposta do socket do localhost enquanto a JVM em Java/Scala processa a query no cluster.
* **Pandas**: O Pandas gasta quase $100\%$ do seu tempo de CPU em funções otimizadas de baixo nível (`algorithms.py:548(factorize_array)` e `ops.py:358(_call_cython_op)`). Isto comprova que não existe overhead de concorrência ou comunicação e o processador executa processamento útil de dados diretamente na RAM através de vetorização.
* **Dask**: O maior gargalo registou-se no método `{method 'acquire' of '_thread.lock' objects}`. Devido à presença do *Global Interpreter Lock (GIL)* do Python, as várias threads locais paralelas criadas pelo Dask entram em colisão e disputa de locks para assumir o interpretador Python, gerando longos tempos de espera passiva.
* **Modin**: O Modin passou a maior parte do seu tempo no método `{method 'copy' of 'numpy.ndarray' objects}`. A necessidade de fatiar o DataFrame local em pequenas partições e coordená-las gera um número elevado de cópias físicas de memória e manipulação de metadados (`partition.py:76(apply)` e `partition.py:95(call_queue_closure)`), sobrecarregando a CPU local.
* **Joblib**: O Joblib passou o seu tempo de execução bloqueado em chamadas `{built-in method time.sleep}`. O processo principal é forçado a adormecer ciclicamente pelo sistema operativo para aguardar que os processos trabalhadores concluam o processamento e serializem os resultados.

---

## 2. Análise Comparativa: Com Profiling vs. Sem Profiling

A comparação entre os resultados do Benchmarking puro e os do Diagnóstico expõe de forma explícita o conceito teórico do **Efeito de Observador (*Observer Effect / Profiling Overhead*)**:

| Operação (Escala Média) | Pandas Sem Profiling (s) | Pandas Com Profiling (s) | PySpark Sem Profiling (s) | PySpark Com Profiling (s) | Dask Sem Profiling (s) | Dask Com Profiling (s) |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **Value Counts** | 0.015 | 0.093 | 0.056 | 0.128 | 0.160 | 0.331 |
| **GroupBy** | 0.039 | 0.130 | 0.059 | 0.092 | 0.289 | 0.288 |

### 3.1. Diferenças Críticas Observadas:
1. **O Efeito de Observador (Profiling Overhead)**:
   * A execução **Com Profiling** é consideravelmente mais demorada do que a execução **Sem Profiling** (ex: o `Value Counts` do Pandas sobe de $0.015\text{ s}$ para $0.093\text{ s}$; o Dask sobe de $0.160\text{ s}$ para $0.331\text{ s}$).
   * **Razão Técnica**: O profiling via `cProfile` exige que o interpretador intercete e registe os metadados de cada micro-chamada a funções ao nível de código e de C subjacente. Esta atividade de monitorização contínua adiciona um atraso sistemático à computação.
2. **A Estabilidade de Funções Relativas**:
   * Embora o profiling altere os tempos de execução absolutos, ele preserva as relações de grandeza e eficiência relativa entre as bibliotecas. O Pandas sequencial vetorizado mantém-se o mais ágil localmente devido ao seu reduzido número de chamadas de interpretador (apenas 591 chamadas).
   * O Dask, pelo contrário, gera um grafo massivo de **214.276 chamadas de funções** locais quer em escala pequena quer em média. Embora sem profiling o Dask consiga mascarar parte deste overhead em threads rápidas, a interceção forçada do cProfile na totalidade dessas micro-chamadas torna o atraso ainda mais notório.

### 3.2. Conclusão Metodológica
* **Para Medição de Produção**: Devem usar-se exclusivamente os resultados **Sem Profiling** (Notebook 1), pois representam a verdadeira velocidade física útil sentida pelo utilizador ou cliente.
* **Para Diagnóstico e Engenharia de Código**: Devem usar-se os dados **Com Profiling** (Notebook 2). Embora mais lentos, os dados expõem as dinâmicas internas (onde a CPU está a gastar tempo: se a mover RAM, a esperar por rede, ou a competir pelo GIL), fornecendo as pistas cruciais para otimização arquitetural.